### Before start
- The main content of the following scripts were adapted from ISC tutorial notebook by Samuel A. Nastase. For more details see: https://github.com/snastase/isc-tutorial/blob/master/isc_tutorial.ipynb
- We only used static and pairwise approach to calculate ISFC in the current paper. For other methods, please check the original tutorial.
- Before running the analysis, you might want to read some theoretical materials about ISC.
Please check article by Juha Lahnakoski and Luke Chang on: https://naturalistic-data.org/content/Intersubject_Correlation.html



Click *Open in playground mode* to interactively run and edit cells (you may need to sign into a Google account), and use *File* > *Save a copy in Drive...* or *Save a copy in GitHub...* to save your changes. To execute a code cell, click in that cell and press _Shift_ + _Enter_ (or _Shift_ + _Return_ on a Mac). The first time you run a code cell in playground mode, you may receive a warning and a prompt to reset runtimes; click _Run anyway_ followed by _Yes_.


We used basic ISC functionality without the full BrainIAK package. We'll download `isc_standalone.py` from the [GitHub repository](https://github.com/snastase/isc-tutorial) and load the necessary modules locally. If you've cloned the `isc-tutorial` GitHub repository locally, this step is not necessary (as the local directory for the repository already contains `isc_standalone.py`).

In [1]:
# Download the standalone module of isc analysis
from urllib.request import urlretrieve
urlretrieve('https://github.com/snastase/isc-tutorial/'
            'raw/master/isc_tutorial/isc_standalone.py', 'isc_standalone.py');
            #if this link does not work, download isc_standalone.py from the same folder as this script



Import the relevant functions from `isc_standalone`.

In [ ]:
from isc_standalone import (isc, isfc, bootstrap_isc, _check_isc_input,
                            squareform_isfc, _threshold_nans, _check_group_assignment,
                            _get_group_parameters, _permute_one_sample_iscs,
                            _permute_two_sample_iscs, p_from_null,
                            compute_summary_statistic, load_images,
                            load_boolean_mask, mask_images,
                            MaskedMultiSubjectData)

Finally, we'll load several other useful Python modules.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, pearsonr, zscore
from scipy.spatial.distance import squareform
from statsmodels.stats.multitest import multipletests
import nibabel as nib

from scipy import io
import pandas as pd

import itertools as it
from google.colab import files

In [ ]:
# input data here [your time series]
# it should be the time series file after running preprocessing for each subgroup of images
data = np.load("asd_de.npy")

# check the shape of your data, should be n_timepoints * n_ROIs * n_subjects
print(data.shape)

When there are three or more subjects, we can compute ISCs/ISFCs using either the pairwise approach (`pairwise=True`), where we compute ISCs/ISFCs between each pair of subjects/regions, or the leave-one-out (`pairwise=False`) approach, where we compute ISCs/ISFCs between each subject and the average time series of other subjects.

### Pairwise approach
Now we'll run the full-scale ISC analysis across all ROIs and subjects using the pairwise approach. For a given ROI, the correlations between each pair of subjects are represented in a vector of length
```
n_subjects * (n_subjects - 1) / 2
```
or 190 pairs for 20 subjects. This vector of pairs corresponds to the off-diagonal values of a symmetric subjects-by-subjects correlation matrix.



In [ ]:
# Pairwise approach across all subjects and voxels
iscs = isc(data, pairwise=True)

# Check shape of output ISC values
print(f"ISC values shape = {iscs.shape} \ni.e., {iscs.shape[0]} "
      f"pairs and {iscs.shape[1]} voxels"
      f"\nMinimum ISC = {np.amin(iscs):.3f}; "
      f"maximum ISC = {np.amax(iscs):.3f}")

For a given ROI, we can convert the vector of pairs to the full correlation matrix for visualization.

In [ ]:
# Visualize the correlation matrix for one voxel
isc_matrix = squareform(iscs[:, 0])
np.fill_diagonal(isc_matrix, 1)

# change 20 to the n_subject you have in this dataset
sns.heatmap(isc_matrix, cmap="RdYlBu_r", vmin=-1, vmax=1, square=True,
            xticklabels=range(1, 23), yticklabels=range(1, 20))
plt.xticks(np.arange(0, 20))
plt.yticks(np.arange(0, 20))
plt.xlabel('subjects')
plt.ylabel('subjects')
plt.show()

### Input types
Currently, we're submitting a list of numpy arrays to BrainIAK's `isc` function where each item in the list is a subject's response time course over some number of voxels. Alternatively, we could stack subjects along the 3rd dimension (`np.dstack`) into a single 3-dimensional numpy array and submit this to the `isc` function. If the `isc` function receives a single numpy array, it will assume that the last dimension indexes subjects.

In [ ]:
# Input a list of subjects (same as before)
iscs = isc(data, pairwise=False)

# Stack subjects in 3rd-dimension and recompute ISC
data_stack = np.dstack(data)
print(f"Stacked data shape = {data_stack.shape}"
      f"\ni.e., {data_stack.shape[0]} time points, {data_stack.shape[1]} "
      f"voxels, and {data_stack.shape[2]} subjects")

# Input stacked numpy array
iscs_from_stack = isc(data_stack, pairwise=False)

# Make sure the ISC outputs are the same
assert np.array_equal(iscs, iscs_from_stack)

### Summary statistics
Rather than returning ISC values for each pair of subject (in the pairwise approach) or each left-out subject (in the leave-one-out approach), we can use the `summary_statistic` argument to output either the mean or median across the values. Note that by default `summary_statistic=False`. If we request the mean ISC value, the `isc` function will internally apply the Fisher *z*-transformation (`np.arctanh`) prior to computing the mean, then apply the inverse Fisher *z*-transformation (`np.tanh`) to the mean value.

In [ ]:
# Compute mean pairwise ISC
iscs = isc(data, pairwise=True, summary_statistic='mean')

print(f"ISC values shape = {iscs.shape} \ni.e., the mean value across "
      f"pairwise subjects for {iscs.shape[0]} roi(s)"
      f"\nMean ISC for first roi = {iscs[0]:.3f}")

# Compute median pairwise ISC
iscs = isc(data, pairwise=True, summary_statistic='median')

print(f"ISC values shape = {iscs.shape} \ni.e., the median value across "
      f"pairwise subjects for {iscs.shape[0]} roi(s)"
      f"\nMedian ISC for first roi = {iscs[0]:.3f}")

## Statistical tests
BrainIAK provides several nonparametric statistical tests for ISC analysis. Nonparametric tests are preferred due to the inherent correlation structure across ISC values—each subject contributes to the ISC of other subjects, violating assumptions of independence required for standard parametric tests (e.g., *t*-test, ANOVA). The nonparametric statistical tests discussed below return the actual observed ISC values, *p*-values, and the resampling distribution (the bootstrap hypothesis test also returns confidence intervals around the observed ISC statistic).

### Bootstrap hypothesis test (for one-sample testing)
For one-sample tests, we can resample subjects with replacement to construct a bootstrap distribution around our observed ISC statistic ([Chen et al., 2016](https://doi.org/10.1016/j.neuroimage.2016.05.023)). This method was proved to show least false positive results for one-sample testing. We can compute confidence intervals around the test statistic using the `ci_percentile` option (default 95%). Hypothesis test is performed by shifting the bootstrap distribution to zero. Note that when constructing the bootstrap distribution using the pairwise approach, subjects (i.e., rows and columns in the subject-by-subject correlation matrix) are sampled with replacement, not pairs (which would disrupt the correlation structure among pairs).

In [ ]:
# load our revised function for bootstrap testing first
def bootstrap_isc(iscs, pairwise=True, summary_statistic='median',
                  n_bootstraps=1000, ci_percentile=95, side='right', random_state=None):


    """
    input:
    iscs: list or ndarray, n_pairs * n_ROIs array

    pairwise: indicator of pairwise or leave-one-out

    summary_statistic: default: 'median' is recommended in non-parametric test
        either 'mean' or 'median'

    n_bootstraps: int, default: 1000
        Number of bootstrap samples (subject-level with replacement)

    ci_percentile: int, default: 95
        Percentile for computing confidence intervals

    side: str, default: 'right' (where distribution >= observed)
        Perform one-sided ('left' or 'right') or 'two-sided' test

    random_state: int or None, default: None
        initial random seed

    output:
    observed: float, median (or mean) ISC value

    ci: tuple, bootstrap confidence intervals
        Confidence intervals generated from bootstrap distribution

    p: float, p-value
        p-value based on bootstrap hypothesis test

    distribution: ndarray, n_bootstraps by ROIs
    """

    iscs, n_subjects, n_ROIs = check_isc_input(iscs, pairwise=pairwise)

    # compute summary statistic for obversed ISCs
    observed = compute_summary_statistic(iscs, summary_statistic=summary_statistic, axis=0)


    # build bootstrap distribution
    distribution = []

    # loop through n bootstrap iterations and populate distribution
    for i in np.arange(n_bootstraps):

        # random seed to be deterministically re-randomized at each iteration
        if isinstance(random_state, np.random.RandomState):
            prng = random_state
        else:
            prng = np.random.RandomState(random_state)

        # randomly sample subject IDs with replacement
        subject_sample = sorted(prng.choice(np.arange(n_subjects), size=n_subjects))
        print(subject_sample)

        # sqaureform and shuffle rows/columns of pairwise ISC matrix
        # to retain correlation structure among ISCs, then get triangle
        if pairwise:

            # loop through ROIs
            isc_sample = []
            for ROI_iscs in iscs.T:

                # square the triangle and fill diagonal
                ROI_iscs = squareform(ROI_iscs, force='tomatrix')
                np.fill_diagonal(ROI_iscs, 1)

                # shuffle square correlation matrix and get triangle
                ROI_sample = ROI_iscs[subject_sample,:][:, subject_sample]
                ROI_sample = squareform(ROI_sample, checks=False)

                # censor off-diagonal 1s for same-subject pairs
                ROI_sample[ROI_sample == 1.] = np.NaN

                isc_sample.append(ROI_sample)

            isc_sample = np.column_stack(isc_sample)
            print(isc_sample.shape)

        # compute summary statistic for bootstrap ISCs per ROI
        distribution.append(compute_summary_statistic(
                                isc_sample,
                                summary_statistic=summary_statistic,
                                axis=0))

        # update random state for next iteration
        MAX_RANDOM_SEED = 2**32 - 1
        random_state = np.random.RandomState(prng.randint(0, MAX_RANDOM_SEED))

    # convert distribution to numpy array
    distribution = np.array(distribution)

    # compute CIs of median from bootstrap distribution (default: 95%)
    ci = (np.percentile(distribution, (100- ci_percentile)/2, axis=0),
          np.percentile(distribution, ci_percentile + (100 - ci_percentile)/2,
                          axis=0))

    # shift bootstrap distribution to 0 for hypothesis test
    shifted = distribution - observed

    # get p-value for actual median from shifted distribution
    p = p_from_null(observed, shifted, side=side, exact=False, axis=0)

    return observed, ci, p, distribution

In [ ]:
# Compute ISCs and then run bootstrap hypothesis test on ISCs
iscs = isc(data, pairwise=True, summary_statistic=None)

#change n_bootstraps to what you can wait, iteration at 1000 or more is recommended
observed, ci, p, distribution = bootstrap_isc(iscs, pairwise=True,
                                              ci_percentile=95,
                                              summary_statistic='median',
                                              n_bootstraps=1000)

In [ ]:
# Inspect shape of null distribution
print(f"Null distribution shape = {distribution.shape}"
      f"\ni.e., {distribution.shape[0]} bootstraps "
      f"and {distribution.shape[1]} roi")

# Get actual ISC value and p-value for first voxel
print(f"Actual observed ISC value for first roi = {observed[0]:.3f},"
      f"\np-value from bootstrap hypothesis test = {p[0]:.3f}")

In [ ]:
# save the results as npz file containing multiple numpy arrays
np.savez("data_bootstrap", observed=observed, ci=ci, p=p, distribution=distribution)

### Permutation test (for two-sample testing)
We can use a permutation test to statistically evaluate one- or two-sample tests ([Chen et al., 2016](https://doi.org/10.1016/j.neuroimage.2016.05.023)). In the current paper, we used permutation test only for group comparision as suggestions of Chen's study. Depends on your research question, permutation test can also be used for one-sample testing.

In the case of a one-sample test, we use a sign-flipping (-1, +1) approach applied to the observed ISC. For a two-sample test, we supply a `group_assignment` list containing the group labels for each subject. The order of the group assignment list must match the order in which the subjects are supplied to the `isc` function. At each iteration, we randomly reassign the group labels, then compute the test statistic. In the one-sample test, there are `2**n_subjects` number of possible permutations, while in the the two-sample test, there are `n_subjects!` number of possible permutations. In both cases, if the requested number of permutations equals or exceeds the exhaustive list of permutations, an exact test is performed using all possible permutations. However, in most cases the number of subjects will yield an a prohibitively large number of permutations, in which case a Monte Carlo approximate permutation test is used instead of an exact test.

In [ ]:
# load our revised function for permutation testing first
def permutation_isc(iscs, group_assignment=None, pairwise=False,
                    summary_statistic='median', n_permutations=1000,
                    side='right', random_state=None):

        # Standardize structure of input data
    iscs, n_subjects, n_voxels = _check_isc_input(iscs, pairwise=pairwise)

    # Check for valid summary statistic
    if summary_statistic not in ('mean', 'median'):
        raise ValueError("Summary statistic must be 'mean' or 'median'")

    # Check match between group labels and ISCs
    group_assignment = _check_group_assignment(group_assignment,
                                               n_subjects)

    # Get group parameters
    group_parameters = _get_group_parameters(group_assignment, n_subjects,
                                             pairwise=pairwise)

    # Set up permutation type (exact or Monte Carlo)
    if group_parameters['n_groups'] == 1:
        if n_permutations < 2**n_subjects:
            print("One-sample approximate permutation test using "
                        "sign-flipping procedure with Monte Carlo resampling.")
            exact_permutations = None
        else:
            print("One-sample exact permutation test using "
                        "sign-flipping procedure with 2**{0} "
                        "({1}) iterations.".format(n_subjects,
                                                   2**n_subjects))
            exact_permutations = list(it.product([-1, 1], repeat=n_subjects))
            n_permutations = 2**n_subjects

    # Check for exact test for two groups
    else:
        if n_permutations < np.math.factorial(n_subjects):
            print("Two-sample approximate permutation test using "
                        "group randomization with Monte Carlo resampling.")
            exact_permutations = None
        else:
            print("Two-sample exact permutation test using group "
                        "randomization with {0}! "
                        "({1}) iterations.".format(
                                n_subjects,
                                np.math.factorial(n_subjects)))
            exact_permutations = list(it.permutations(
                np.arange(len(group_assignment))))
            n_permutations = np.math.factorial(n_subjects)

    # If one group, just get observed summary statistic
    if group_parameters['n_groups'] == 1:
        observed = compute_summary_statistic(
                        iscs,
                        summary_statistic=summary_statistic,
                        axis=0)[np.newaxis, :]

    # If two groups, get the observed difference
    else:
        observed = (compute_summary_statistic(
                        iscs[group_parameters['group_selector'] ==
                             group_parameters['group_labels'][0], :],
                        summary_statistic=summary_statistic,
                        axis=0) -
                    compute_summary_statistic(
                        iscs[group_parameters['group_selector'] ==
                             group_parameters['group_labels'][1], :],
                        summary_statistic=summary_statistic,
                        axis=0))
        observed = np.array(observed)

    # Set up an empty list to build our permutation distribution
    distribution = []

    # Loop through n permutation iterations and populate distribution
    for i in np.arange(n_permutations):

        # Random seed to be deterministically re-randomized at each iteration
        if exact_permutations:
            prng = None
        elif isinstance(random_state, np.random.RandomState):
            prng = random_state
        else:
            prng = np.random.RandomState(random_state)

        # If one group, apply sign-flipping procedure
        if group_parameters['n_groups'] == 1:
            isc_sample = _permute_one_sample_iscs(
                            iscs, group_parameters, i,
                            pairwise=pairwise,
                            summary_statistic=summary_statistic,
                            exact_permutations=exact_permutations,
                            prng=prng)

        # If two groups, set up group matrix get the observed difference
        else:
            isc_sample = _permute_two_sample_iscs(
                            iscs, group_parameters, i,
                            pairwise=pairwise,
                            summary_statistic=summary_statistic,
                            exact_permutations=exact_permutations,
                            prng=prng)

        # Tack our permuted ISCs onto the permutation distribution
        distribution.append(isc_sample)

        # Update random state for next iteration
        if not exact_permutations:
            random_state = np.random.RandomState(prng.randint(
                                                    0, MAX_RANDOM_SEED))

    MAX_RANDOM_SEED = 2**32 - 1

    # Convert distribution to numpy array
    distribution = np.array(distribution)

    # Get p-value for actual median from shifted distribution
    if exact_permutations:
        p = p_from_null(observed, distribution,
                        side=side, exact=True,
                        axis=0)
    else:
        p = p_from_null(observed, distribution,
                        side=side, exact=False,
                        axis=0)

    return observed, p, distribution

## ISFC analysis
Rather than computing ISCs for corresponding voxels across participants, we can instead compute ISCs between all voxels/ROIs to measure functional integration (i.e., connectivity). This method is called intersubject functional correlation (ISFC) analysis ([Simony et al., 2016](https://doi.org/10.1038/ncomms12141)). Using the `vectorize_isfcs` option, we can either return a tuple containing the condensed off-diagonal ISFC values and the diagonal ISC values or the square (redundant) ISFC values. If `vectorize_isfcs=True` (the default), the first array in the tuple contains the off-diagonal ISFC values for each pair of voxels as condensed by `scipy.spatial.distance.squareform` and is shaped `n_subjects` (or `n_pairs`) by `n_connections` where
```
n_connections = n_voxels * (n_voxels - 1) / 2

(here, n_voxels can also be n_ROIs if ROI-based analysis)
```
The second array in the tuple is the diagonal values shaped `n_subjects` (or `n_pairs`) by `n_voxels`. If `vectorize_isfcs=False`, we get a 3-dimensional array containing the square (redundant) ISFC and ISC values, shaped `n_subjects` (or `n_pairs`) by `n_voxels` by `n_voxels`.  If a `summary_statistic` is supplied, or only two subjects are input, the singleton first dimension is removed.

If we have two groups we expect to have different ISC values, we must supply a `group_assignment` list. In the case of two groups, we compute the difference between the `summary_statistic` for each group. In the pairwise approach, we compute differences between the `summary_statistic` for within-group correlations, and ignore the between group correlations in the full subject-by-subject correlation matrix containing both groups. Furthermore, permutations are applied to subjects (i.e., rows and columns in the subject-by-subject correlation matrix) and not to pairs (which would disrupt the correlation structure among pairs).

In [ ]:
de_nt = np.load("de_nt_new_ts.npy")
de_nt.shape # N_subject = 25

In [ ]:
de_asd = np.load("ts_asd_de_0307.npy")
de_asd.shape # N_subject = 22

In [ ]:
# combine german_nt + asd data
data = np.concatenate((de_nt, de_asd), axis=2)
data.shape

In [ ]:
isfcs, iscs = isfc(data, pairwise=True, vectorize_isfcs=True)

In [ ]:
# code for two-sample permutation (we used pairwise, median)

# ⚠️important: the n_subject for group 1 and group 2 should be aligned with the input data order
#               otherwise the results won't be correct
group_assignment = [1]*25 + [2]*22  #change 25/22 to your N in each clinical group (autism or neurotypical)


observed, p, distribution = permutation_isc(isfcs,
                                            group_assignment=group_assignment,
                                            pairwise=True,
                                            summary_statistic='median',
                                            side='right',
                                            n_permutations=5000)
#change n_permutations to smaller number if for testing otherwise will take very long (1h+)

In [ ]:
distribution.shape  #should be (n_permutations, n_ROI*(n_ROI-1)/2)

In [ ]:
np.savez("data_two_sample_permutation", observed=observed, p=p, distribution=distribution)

repeat above process for 2nd dataset, 3rd, 4th... & then calculate replication rate in the next session

**save isfcs seprately for visualization purporse**

In [ ]:
# compute isfcs per group
isfcs_asd, iscs = isfc(fin_asd_cut, pairwise=True, vectorize_isfcs=True, summary_statistic='median')

In [ ]:
# compute isfcs per group
isfcs_asd_de, iscs = isfc(de_asd_cut, pairwise=True, vectorize_isfcs=True, summary_statistic='median')

In [ ]:
# compute isfcs per group
isfcs_nt_de, iscs = isfc(de_nt_cut, pairwise=True, vectorize_isfcs=True, summary_statistic='median')

In [ ]:
# compute isfcs per group
isfcs_nt, iscs = isfc(fin_nt_cut, pairwise=True, vectorize_isfcs=True, summary_statistic='median')

In [ ]:
isfc_all = pd.DataFrame(data={'de_nt': isfcs_nt_de.tolist(), 'de_asd': isfcs_asd_de.tolist(), 'fin_nt': isfcs_nt.tolist(), 'fin_asd': isfcs_asd.tolist()})

Alternatively, we can retain the (redundant) structure of the ISFC matrices using `vectorize_isfcs=False` to yield a 3-dimensional array of shape `n_subjects` by `n_voxels/n_ROIs` by `n_voxels/n_ROIs`:

We can also supply a `summary_statistic` to collapse the ISFC values over left-out subjects or pairs of subjects:

We can use the `brainiak.isc.squareform_isfc` convenience function to convert between the condensed representation of ISFCs (with ISCs) and the square (redundant) representation of ISFCs. This function mimics `scipy.spatial.distance.squareform`, but retains the diagonal ISC values.

*please note, in our paper we did not retain ISC values (the diagonal values of ISFC squareform matrix), feel free to keep it if you want to model ISC and ISFC together*

In [ ]:
# convert square ISFCs to condensed ISFCs (and ISCs), and vice versa
# this function mimics scipy.spatial.distance.squareform
def squareform_isfc(isfcs, iscs=None):

    """
    input:
    isfcs: ndarray (n_pairs * n_ROIs * n_ROIs)
        Either condensed or redundant ISFC values

    iscs: ndarray, optional
        Diagonal ISC values, required when input is condensed

    output:
    isfcs: ndarray or tuple of ndarrays
        If condensed ISFCs are passed, a single redundant (square) ISFC array is returned;
        if redundant ISFCs are passed, both a condensed off-diagonal ISFC array and
        the diagonal ISC values are returned;
        shape of condensed off-diagonal ISFC array: n_ROIs * (n_ROIs - 1) /2 (e.g., 37128)
    """

    # check if ISFCs are square (redundant)
    if not type(iscs) == np.ndarray and isfcs.shape[-2] == isfcs.shape[-1]:
        if isfcs.ndim == 2:
            isfcs = isfcs[np.newaxis, ...]
        if isfcs.ndim == 3:
            iscs = np.diagonal(isfcs, axis1=1, axis2=2)
            isfcs = np.vstack([squareform(isfc, checks=False)[np.newaxis, :]
                                for isfc in isfcs])
        else:
            raise ValueError("Square (redundant) ISFCs must be square "
                            "with multiple subjects or pairs of subjects"
                            "indexed by the first dimension")

        return isfcs, iscs

    # or else convert from condensed to redundant
    else:
        isfcs_stack = []
        for isfc, isc in zip(isfcs, iscs):
            isfc_sq = squareform(isfc, checks=False)
            np.fill_diagonal(isfc_sq, isc)
            isfcs_stack.append(isfc_sq[np.newaxis, ...])

        isfcs = np.vstack(isfcs_stack)

        return isfcs

In [ ]:
#from brainiak.isc import squareform_isfc

# Start with square (redundant) ISFCs and check shape
isfcs_sq = isfc(data, pairwise=False, vectorize_isfcs=False)
print(f"Square (redundant) ISFCs shape: {isfcs_sq.shape}")

# Convert these directly to condensed ISFCs (and ISCs)
isfcs_c, iscs = squareform_isfc(isfcs_sq)
print(f"Condensed ISFCs shape: {isfcs_c.shape}, "
      f"ISCs shape: {iscs.shape}")

# Convert these directly back to redundant ISFCs
isfcs_r = squareform_isfc(isfcs_c, iscs)
print(f"Converted redundant ISFCs shape: {isfcs_r.shape}")

# Check that they are identical to the original square ISFCs
assert np.array_equal(isfcs_sq, isfcs_r)

Let's confirm that the diagonal of the ISFC matrix represents each voxel correlated with itself across subjects—the conventional ISC described above. We can see that the conventional ISC analysis is in fact a subset of the ISFC analysis.

In [ ]:
# Get ISC values directly from ISFC matrix
isfcs, iscs = isfc(data, pairwise=False, vectorize_isfcs=True)

# Check that these are the same as conventional ISCs
assert np.allclose(iscs, isc(data))

Finally, we can visualize the matrix of mean (or median) ISFC values. If we used `vectorize_isfcs=True`, we'll first need to apply `squareform_isfc` the ISFC (and ISC) values. The diagonal blocks represent the 10 artificial "networks" in our simulated data; the 100 voxels in each network are highly correlated with each other and largely uncorrelated with voxels in other networks.

In [ ]:
isfcs = np.load("isfc_de_nt.npy")

In [ ]:
# Recompute mean ISFCs
#isfcs, iscs = isfc(data, pairwise=False, summary_statistic='mean',
#                   vectorize_isfcs=True)

# Convert these to a square representation
#isfcs = squareform_isfc(isfcs, iscs)

# Visual mean ISFC matrix
plt.matshow(isfcs, cmap="RdYlBu_r", vmin=-0.2, vmax=0.65)
plt.grid(False)
np.fill_diagonal(isfcs, iscs)
#plt.xticks(np.arange(0, 1001, 100)[1:], np.arange(100, 1001, 100),
#           rotation=45)
plt.gca().xaxis.tick_bottom()
plt.gca().xaxis.set_label_position('bottom')
#plt.yticks(np.arange(0, 1001, 100)[1:], np.arange(100, 1001, 100))
plt.title("isfc_median of German NT group")
plt.xlabel('ROIs')
plt.ylabel('ROIs')
ax = plt.gca()
#plt.colorbar()
plt.colorbar(fraction=0.046, pad=0.04);

## Calculate replication rate
After running two-sample permutation tests and saved the results as npy files, load again here and calculate the replication rate of group difference (neurotypical-autism) across two datasets

*here, we defined 'replicated' by same ROI-to-ROI pairwise ISFCs that are significant at confidence level 0.01/0.05. For voxel-wise analysis, please define 'replicated' results yourself*.

In [ ]:
# matlab code for calculating replication rate across permutation test (nt-asd) results
# transferred to python below

import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform

# Load MATLAB .mat files of permutation tests for each dataset --> load permutation tests' npy files
DE = np.load('de_nt_asd.npy') #change to your output of two-sample permutation tests
FI = np.load('fin_nt_asd.npy')

# Extract data (adjust keys as needed for your npy object)
de_distribution = DE['de']['distribution'][0, 0]
fi_distribution = FI['fin']['distribution'][0, 0]

de_observed = DE['de']['observed'][0, 0]
fi_observed = FI['fin']['observed'][0, 0]

# Define threshold
pval = 0.025  # Two-sided p-value at 0.01
thrs = [
    np.quantile(de_distribution, 1 - pval),
    np.quantile(fi_distribution, 1 - pval)
]

# Thresholding & masking
difmat = squareform(de_observed * (np.abs(de_observed) > thrs[0]))
difmat2 = squareform(fi_observed * (np.abs(fi_observed) > thrs[1]))

# Overlap matrix
replMat = np.sign(difmat) + np.sign(difmat2)
sigInd = difmat != 0  # use German (DE) results only as mask

# Replication rate
replication_rate = 100 * np.sum(np.abs(replMat[sigInd]) == 2) / np.sum(sigInd)

# Plotting
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

im0 = axs[0].imshow(difmat, cmap='bwr', vmin=-0.1, vmax=0.1)
axs[0].set_title('German sig. pairs')

im1 = axs[1].imshow(difmat2, cmap='bwr', vmin=-0.1, vmax=0.1)
axs[1].set_title('Finnish sig. pairs')

im2 = axs[2].imshow(replMat, cmap='bwr', vmin=-2, vmax=2)
axs[2].set_title(f'Replication rate = {replication_rate:.2f}%')

plt.tight_layout()
plt.show()

## Correcting for multiple tests*
Evaluating the statistical significance of an ISC analysis across many voxels will result in many false positives unless we somehow control for the large number of statistical tests. Here we'll use two simple approaches for correcting for multiple tests. In the first approach, we'll account for multiple tests by controlling the expected proportion of false positives or false discovery rate (FDR; [Benjamini & Hochberg, 1995](https://www.jstor.org/stable/2346101); [Benjamini & Yekutieli, 2001](https://www.jstor.org/stable/2674075); [Genovese et al., 2002](https://doi.org/10.1006/nimg.2001.1037)). In the second approach, we'll control the family-wise error rate (FWER) by constructing a null distribution from the maximum ISC value across all voxels at each iteration of a randomization test ([Nichols & Holmes, 2002](https://doi.org/10.1002/hbm.1058)).


***Please be aware that we did not do this in our manuscript since we are ROI-based analysis, feel free to do it in your dataset if it's voxel-based analysis.***

### Controlling FDR
To control FDR, we'll use the the `multipletests` function from the StatsModels Python package. This returns an array of *q*-values, which are typically interpreted as FDR-corrected *p*-values. By thresholding uncorrected and corrected *p*- and *q*-values, we can determine how many voxels survived correction for multiple tests.

In [ ]:
# Get q-values (i.e., FDR-controlled p-values) using statsmodels
q = multipletests(p, method='fdr_by')[1]

# We can also convert these q-values to z-values
z = np.abs(norm.ppf(q))

# Also get significant voxels with and without correction
corrected = q[np.newaxis, :] < .05
uncorrected = p[np.newaxis, :] < .05

# Count significant voxels before and after correction
print(f'{np.sum(uncorrected)} "significant" voxels before correction for '
      f"multiple tests; {np.sum(corrected)} significant voxels after "
      f"controlling FDR at .05")

Finally, we can visualize the voxel time series for an example subject, the ISC values across subjects, and which voxels are considered significant before and after controlling FDR at .05. Note that before correction even some of the noisy voxels are considered to have significant ISC; however, after correction, the number of significant noisy voxels is reduced.

In [ ]:
# Set up grid of subplots for visualizing voxel values and significance
fig, (ax0, ax1, ax2, ax3) = plt.subplots(nrows=4, figsize=(12, 8),
                                         sharex=True,
                                         gridspec_kw={'height_ratios':
                                                      [300, 190, 20, 20]})

# Visualize data for first subject where half of voxels are noisy
ax0.matshow(noisy_data[..., 0], cmap='RdYlBu_r', vmin=-3, vmax=3)
ax0.grid(False)
ax0.set_ylabel('time points')
ax0.set_title('response time series for example subject', y=1)

# Visualize ISC values across all pairs of subjects
ax1.matshow(iscs, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax1.grid(False)
ax1.set_ylabel('pairs of subjects')
ax1.set_title('ISC values for all pairs of subjects', y=1)

# Visualize uncorrected and corrected significant voxels
ax2.matshow(np.repeat(uncorrected, 20, axis=0),
            cmap='viridis',vmin=0, vmax=1)
ax2.grid(False)
ax2.set_yticks([])
ax2.set_title('uncorrrected "significant" voxels (yellow)')

ax3.matshow(np.repeat(corrected, 20, axis=0),
            cmap='viridis',vmin=0, vmax=1)
ax3.grid(False)
ax3.set_xlabel('voxels')
ax3.xaxis.tick_bottom()
ax3.set_yticks([])
ax3.set_title('FDR-corrrected significant voxels (yellow)')
plt.tight_layout()

### Controlling FWER
To strictly control the FWER, one method is to construct a null distribution of maximum ISC statistics across all voxels. First we'll use the `permutation_isc` function to run a one-sample two-sided permutation test using a sign-flipping procedure, which returns *p*-values and a null distribution.

In [ ]:
# Compute ISCs and then run two-sample permutation test on ISCs
iscs = isc(data, pairwise=True, summary_statistic=None)
observed, p, distribution = permutation_isc(iscs, pairwise=True,
                                            summary_statistic='mean',
                                            n_permutations=1000)

Next, we'll write a simple function that takes a null distribution with multiple voxels, and aggregates the maximum ISC value across all voxels for each null sample.

In [ ]:
# Loop through null distribution and get maximum value across voxels
def get_maxima(distribution):
  max_distribution = []
  for i in distribution:
    max_isc = np.amax(i)
    max_distribution.append(max_isc)
  max_distribution = np.array(max_distribution)
  return max_distribution

After we create a null distribution of maximum statistics, any voxel with an ISC value in the top 5% of distribution can be considered significant. Here we compute *p*-values from the null distribution of maximum statistics using a two-sided test.

In [ ]:
# Create null distribution of maximum ISCs across all voxels
max_distribution = get_maxima(distribution)

# Broadcast our max distribution across all 1000 voxels
max_distribution = np.repeat(max_distribution[:, np.newaxis], 273, axis=1)

# Get the summary statistic (median) for our actual ISC values
# since we set summary_statistic=None above
observed = np.median(iscs, axis=0)[np.newaxis, :]

# Evaluate whether observed ISCs land in the tail of the max distribution
p_max = ((np.sum(np.abs(max_distribution) >= np.abs(observed), axis=0) + 1) /
          float((len(max_distribution) + 1)))[np.newaxis, :]

# Get p-values less than .05 (corrected for multiple tests)
corrected = p_max < .05

As with the FDR approach,  we can visualize the data and the voxels marked as significant before and after correction for multiple tests. This method of correction for multiple tests is considerably more conservative.

In [ ]:
# Set up grid of subplots for visualizing voxel values and significance
fig, (ax0, ax1, ax2, ax3) = plt.subplots(nrows=4, figsize=(12, 8),
                                         sharex=True,
                                         gridspec_kw={'height_ratios':
                                                      [300, 190, 20, 20]})

# Visualize data for first subject where half of voxels are noisy
ax0.matshow(data[..., 0], cmap='RdYlBu_r', vmin=-3, vmax=3)
ax0.grid(False)
ax0.set_ylabel('time points')
ax0.set_title('response time series for example subject', y=1)

# Visualize ISC values across all pairs of subjects
ax1.matshow(iscs, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax1.grid(False)
ax1.set_ylabel('pairs of subjects')
ax1.set_title('ISC values for all pairs of subjects', y=1)

# Visualize uncorrected and corrected significant voxels
ax2.matshow(np.repeat(uncorrected, 20, axis=0),
            cmap='viridis',vmin=0, vmax=1)
ax2.grid(False)
ax2.set_yticks([])
ax2.set_title('uncorrrected "significant" voxels (yellow)')

ax3.matshow(np.repeat(corrected, 20, axis=0),
            cmap='viridis',vmin=0, vmax=1)
ax3.grid(False)
ax3.set_xlabel('voxels')
ax3.xaxis.tick_bottom()
ax3.set_yticks([])
ax3.set_title('FWER-corrrected significant voxels (yellow)')
plt.tight_layout()

Note that there are many other ways to correct for multiple tests, such as using cluster-extent thresholding.